In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm

# Problem 2

### Importing data

In [2]:
FF = pd.read_csv('FF_allhistory.csv')

SR_total = pd.read_excel('HW3_RD_ANALYSIS.xlsx', sheet_name="s.hedge-StatArb03-'07-'09 ")
SR_long  = pd.read_excel('HW3_RD_ANALYSIS.xlsx', sheet_name="s.hedge-StatArb03-'07-'09 long")
SR_short = pd.read_excel('HW3_RD_ANALYSIS.xlsx', sheet_name="s.hedge-StatArb03-'07-'09 short")

In [3]:
FF['d'] = pd.to_datetime(FF['d'])
SR_total['Dates'] = pd.to_datetime(SR_total['Dates'])
SR_long['Dates']  = pd.to_datetime(SR_long['Dates'])
SR_short['Dates'] = pd.to_datetime(SR_short['Dates'])

FF.rename(columns={'d': 'Dates'}, inplace=True)

FF = FF.set_index('Dates')
SR_total = SR_total.set_index('Dates')
SR_long  = SR_long.set_index('Dates')
SR_short = SR_short.set_index('Dates')

In [4]:
ret_col = SR_total.columns[0]
ret_col_long = SR_long.columns[0]
ret_col_short = SR_short.columns[0]

DF       = SR_total.join(FF, how='inner')
DF_long  = SR_long.join(FF, how='inner')
DF_short = SR_short.join(FF, how='inner')

print(f'Total: {DF.shape}, Long: {DF_long.shape}, Short: {DF_short.shape}')
DF.head()

Total: (756, 6), Long: (755, 6), Short: (755, 6)


,WRTC,mktrf,smb,hml,rf,umd
Dates,,,,,,
2007-01-03,0.010548,-0.0005,0.0003,0.0020,0.0002,-0.0045
2007-01-04,0.007887,0.0016,0.0023,-0.0051,0.0002,-0.0056
2007-01-05,0.004962,-0.0073,-0.0094,-0.0029,0.0002,0.0005
2007-01-08,-0.009987,0.0024,-0.0009,0.0004,0.0002,0.0032
2007-01-09,-0.001245,0.0000,0.0029,-0.0031,0.0002,0.0025


## Part (a): Annualized Return, Volatility, and Sharpe Ratio

In [5]:
n_days = 252

def perf_stats(df, ret_col_name, label):
    r = df[ret_col_name]
    excess = r - df['rf']
    ann_ret = r.mean() * n_days
    ann_vol = r.std() * np.sqrt(n_days)
    ann_excess_vol = excess.std() * np.sqrt(n_days)
    ann_rf  = df['rf'].mean() * n_days
    sharpe  = (ann_ret - ann_rf) / ann_excess_vol
    print(f'--- {label} ---')
    print(f'  Annualized Return:     {ann_ret*100:7.2f}%')
    print(f'  Annualized Volatility: {ann_vol*100:7.2f}%')
    print(f'  Sharpe Ratio:          {sharpe:7.4f}')
    print()

perf_stats(DF,       ret_col,       'Overall')
perf_stats(DF_long,  ret_col_long,  'Long Half')
perf_stats(DF_short, ret_col_short, 'Short Half')

--- Overall ---
  Annualized Return:       26.05%
  Annualized Volatility:   34.62%
  Sharpe Ratio:           0.6900

--- Long Half ---
  Annualized Return:      -37.94%
  Annualized Volatility:   39.81%
  Sharpe Ratio:          -1.0069

--- Short Half ---
  Annualized Return:       63.67%
  Annualized Volatility:   35.08%
  Sharpe Ratio:           1.7536



## Part (b): CAPM Regression

In [7]:
def capm_regression(df, ret_col_name, label):
    y = df[ret_col_name] - df['rf']
    X = sm.add_constant(df['mktrf'])
    nw_lags = int(np.floor(4 * (len(y)/100)**(2/9)))
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})
    print(f'=== CAPM (Newey-West): {label} ===')
    print(model.summary())
    print(f'\nAlpha (daily):     {model.params["const"]:.6f}  (t={model.tvalues["const"]:.3f}, p={model.pvalues["const"]:.4f})')
    print(f'Alpha (annualized):{model.params["const"]*252*100:.2f}%')
    print(f'Beta:              {model.params["mktrf"]:.4f}  (t={model.tvalues["mktrf"]:.3f}, p={model.pvalues["mktrf"]:.4f})')
    print(f'R-squared:         {model.rsquared:.4f}')
    sig = 'statistically significant' if model.pvalues['const'] < 0.05 else 'NOT statistically significant'
    print(f'Alpha is {sig} at 5% level (Newey-West).')
    print()
    return model

capm_total = capm_regression(DF,       ret_col,       'Overall')

=== CAPM (Newey-West): Overall ===
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                  0.008049
Date:                Sun, 08 Mar 2026   Prob (F-statistic):              0.929
Time:                        17:37:44   Log-Likelihood:                 1819.6
No. Observations:                 756   AIC:                            -3635.
Df Residuals:                     754   BIC:                            -3626.
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.